# Customer Call List Data Cleaning with Pandas

This notebook cleans a customer call-list spreadsheet using **Pandas**. The steps include removing duplicate records and an unnecessary column, standardizing names and phone numbers, splitting address fields, normalizing contact flags, removing records that should not be contacted, and resetting the index.

## Before you run it

1. Install the dependencies with `pip install pandas openpyxl`.
2. Place `Customer Call List.xlsx` in the same folder as this notebook, or update the input path in the first code cell.
3. Run the cells from top to bottom.

The final cleaned dataset is saved as `Customer Call List Cleaned.xlsx`.

In [ ]:
from pathlib import Path
import pandas as pd

input_file = Path("Customer Call List.xlsx")
output_file = Path("Customer Call List Cleaned.xlsx")

df = pd.read_excel(input_file)
df.head()

## 1. Remove duplicate records and unnecessary columns

In [ ]:
df = df.drop_duplicates()
df = df.drop(columns=["Not_Useful_Column"])
df.head()

## 2. Clean customer names

In [ ]:
df["Last_Name"] = (
    df["Last_Name"]
    .str.lstrip("...")
    .str.lstrip("/")
    .str.rstrip("_")
)

df[["Last_Name"]].head()

## 3. Standardize phone numbers

In [ ]:
# Remove punctuation and other non-alphanumeric characters.
df["Phone_Number"] = df["Phone_Number"].str.replace(r"[^a-zA-Z0-9]", "", regex=True)

# Convert values to strings and format 10-digit phone numbers.
df["Phone_Number"] = df["Phone_Number"].apply(str)
df["Phone_Number"] = df["Phone_Number"].apply(
    lambda value: f"{value[:3]}-{value[3:6]}-{value[6:10]}"
)
df["Phone_Number"] = df["Phone_Number"].str.replace("nan--", "", regex=False)
df["Phone_Number"] = df["Phone_Number"].str.replace("Na--", "", regex=False)

df[["Phone_Number"]].head()

## 4. Split the address into separate fields

In [ ]:
df[["Street_Address", "State", "Zip_Code"]] = df["Address"].str.split(",", n=2, expand=True)
df[["Address", "Street_Address", "State", "Zip_Code"]].head()

## 5. Standardize contact flags and handle missing values

In [ ]:
for column in ["Paying Customer", "Do_Not_Contact"]:
    df[column] = (
        df[column]
        .str.replace("Yes", "Y", regex=False)
        .str.replace("No", "N", regex=False)
    )

# Replace known placeholder values and fill missing cells.
df = df.replace("N/a", "")
df = df.replace("NaN", "")
df = df.fillna("")

df[["Paying Customer", "Do_Not_Contact"]].head()

## 6. Remove contacts that should not be called

In [ ]:
df = df[df["Do_Not_Contact"] != "Y"]

# Remove records without a phone number.
df = df[df["Phone_Number"] != ""]

df = df.reset_index(drop=True)
df.head()

## 7. Save the cleaned dataset

In [ ]:
df.to_excel(output_file, index=False)
print(f"Cleaned dataset saved to: {output_file.resolve()}")